In [ ]:
import pandas as pd
start_year = 1995
discount_rate = 0.05
window_length = 15

df = pd.read_csv('../data/wdi_africa_long.csv')
df = df[df['iso3'] != 'LSO'].reset_index(drop=True)  # Lesotho CPI has multiple uncorrected basket/base-year breaks (1999, 2002, 2009)
df['usd_per_lcu'] = 1 / df['exchange_rate']
df['H'] = df['cpi'] / df['exchange_rate']
df

In [ ]:
import numpy as np
import altair as alt
alt.renderers.enable('mimetype')
alt.themes.register('white_bg', lambda: {'config': {'background': 'white'}})
alt.themes.enable('white_bg')

def compute_pi(cdata, t0, T, col='H', rho=0.05):
    win = cdata[(cdata['year'] >= t0) & (cdata['year'] <= T)].sort_values('year')
    v = win[col].to_numpy()
    g = v / v[0] - 1.0
    t = np.arange(1, len(v) + 1, dtype=float)
    dfactors = (1.0 + rho) ** (-t)
    return float(np.sum(g * dfactors))

def _shade_and_text_layers(cdata, col, hline, shades):
    layers = []
    text_rows = []
    if shades and hline is not None:
        for shade in shades:
            y_start, y_end, color = shade[0], shade[1], shade[2]
            label = shade[3] if len(shade) > 3 else None
            seg = cdata[(cdata['year'] >= y_start) & (cdata['year'] <= y_end)].copy()
            seg['hline'] = hline
            layers.append(
                alt.Chart(seg).mark_area(opacity=0.3, color=color).encode(
                    x=alt.X('year:O'),
                    y=alt.Y(f'{col}:Q'),
                    y2=alt.Y2('hline:Q')
                )
            )
            if label is not None and not seg.empty:
                mid_year = int(seg['year'].iloc[len(seg)//2])
                mid_y = (seg[col].mean() + hline) / 2
                text_rows.append({'year': mid_year, 'y': mid_y, 'text': label, 'color': color})
    return layers, text_rows

def country_chart(iso3, start_year=start_year,
                  h_hline=None, shades=None,
                  usd_hline=None, usd_shades=None,
                  usd_y_range=None, h_y_range=None):
    cdata = df[(df['iso3'] == iso3) & (df['year'] >= start_year)].dropna(subset=['usd_per_lcu', 'H'])
    country_name = cdata['country'].iloc[0]

    legend_setting = alt.Legend(title="Series", orient="bottom")

    usd_scale = alt.Scale(domain=list(usd_y_range)) if usd_y_range is not None else alt.Undefined
    h_scale = alt.Scale(domain=list(h_y_range)) if h_y_range is not None else alt.Undefined

    # --- LEFT AXIS (usd_per_lcu) ---
    left_layers = []
    usd_shade_layers, usd_text_rows = _shade_and_text_layers(cdata, 'usd_per_lcu', usd_hline, usd_shades)
    left_layers.extend(usd_shade_layers)

    left_layers.append(
        alt.Chart(cdata).mark_line(point=True).encode(
            x=alt.X('year:O', title='Year'),
            y=alt.Y('usd_per_lcu:Q', title='USD per LCU', scale=usd_scale),
            color=alt.Color('Metric:N', legend=legend_setting),
            tooltip=['year', 'usd_per_lcu']
        ).transform_calculate(Metric="'usd_per_lcu'")
    )

    if usd_hline is not None:
        left_layers.append(
            alt.Chart(pd.DataFrame({'y':[usd_hline]})).mark_rule(color='blue', strokeDash=[4,4]).encode(
                y=alt.Y('y:Q')
            )
        )

    if usd_text_rows:
        left_layers.append(
            alt.Chart(pd.DataFrame(usd_text_rows)).mark_text(fontSize=14, fontWeight='bold').encode(
                x=alt.X('year:O'),
                y=alt.Y('y:Q'),
                text=alt.Text('text:N'),
                color=alt.Color('color:N', scale=None, legend=None)
            )
        )

    left_chart = alt.layer(*left_layers)

    # --- RIGHT AXIS (H) ---
    right_layers = []
    h_shade_layers, h_text_rows = _shade_and_text_layers(cdata, 'H', h_hline, shades)
    right_layers.extend(h_shade_layers)

    right_layers.append(
        alt.Chart(cdata).mark_line(point=True, color='orange').encode(
            x=alt.X('year:O', title='Year'),
            y=alt.Y('H:Q', title='H = CPI / exchange_rate', axis=alt.Axis(orient='right'), scale=h_scale),
            color=alt.Color('Metric:N', legend=legend_setting),
            tooltip=['year', 'H']
        ).transform_calculate(Metric="'H'")
    )

    if h_hline is not None:
        right_layers.append(
            alt.Chart(pd.DataFrame({'y':[h_hline]})).mark_rule(color='red', strokeDash=[4,4]).encode(
                y=alt.Y('y:Q')
            )
        )

    if h_text_rows:
        right_layers.append(
            alt.Chart(pd.DataFrame(h_text_rows)).mark_text(fontSize=14, fontWeight='bold').encode(
                x=alt.X('year:O'),
                y=alt.Y('y:Q'),
                text=alt.Text('text:N'),
                color=alt.Color('color:N', scale=None, legend=None)
            )
        )

    right_chart = alt.layer(*right_layers)

    return alt.layer(left_chart, right_chart).resolve_scale(y='independent').properties(
        title=f"{country_name} (from {start_year})",
        width=500,
        height=350
    )

dza_data = df[(df['iso3']=='DZA') & (df['year']>=start_year)].dropna(subset=['H'])
h_1998 = dza_data.loc[dza_data['year']==1998, 'H'].iloc[0]
usd_1998 = df[(df['iso3']=='DZA') & (df['year']==1998)]['usd_per_lcu'].iloc[0]
pi_red   = compute_pi(dza_data, 1998, 2006, col='H')
pi_green = compute_pi(dza_data, 2006, 2013, col='H')
pi_total = pi_red+pi_green
pi_blue  = compute_pi(dza_data, 1998, 2013, col='usd_per_lcu')
print(f"H_1998 (Algeria)         = {h_1998:.4f}")
print(f"usd_per_lcu_1998 (Algeria) = {usd_1998:.6f}")
print(f"pi_H(1998,2006)            = {pi_red:.4f}")
print(f"pi_H(2006,2013)            = {pi_green:.4f}")
print(f"pi_H(1998,2013)            = {pi_total:.4f}")
print(f"pi_usd_per_lcu(1998,2013)  = {pi_blue:.4f}")

country_chart(
    'DZA',
    h_hline=h_1998,
    shades=[
        (1998, 2006, 'red',   f"{pi_red:.3f}"),
        (2006, 2013, 'green', f"{pi_green:.3f}"),
    ],
    usd_hline=usd_1998,
    usd_shades=[
        (1998, 2013, 'blue', f"{pi_blue:.3f}"),
    ],
    usd_y_range=(0.006, 0.030),
)

In [ ]:
country_chart('KEN')

In [ ]:
def compute_terminal_residuals_wdi(
    df,
    years=15,
    discount_rate=0.05,
    window_mode="rolling", # or "two_non_overlapping",  # 
    col="H",
    start_year=None,
):
    """Mirror of source.data_analysis_functions.compute_terminal_residuals,
    operating directly on the wdi_africa_long-shaped df with columns:
    iso3, country, year, exchange_rate, cpi, usd_per_lcu, H.

    For each country, slides a fixed-horizon window over the series in `col`,
    computes git = v/v[t0] - 1, discounts year-end cashflows back to start,
    and returns the terminal residual pi, normalized by alpha^2.
    If `start_year` is given, df is filtered to year >= start_year before
    any windowing.
    """
    if start_year is not None:
        df = df[df["year"] >= start_year]

    af = (1 - (1 + discount_rate) ** -years) / discount_rate
    out = []

    for iso3, g in df.dropna(subset=[col]).groupby("iso3"):
        g = g.sort_values("year").reset_index(drop=True)
        n = len(g)
        if n < years:
            continue

        if window_mode == "two_non_overlapping":
            starts = list(range(0, n - years + 1, years))[:2]
        elif window_mode == "rolling":
            starts = list(range(0, n - years + 1))
        else:
            raise ValueError("window_mode must be 'two_non_overlapping' or 'rolling'")

        for i0 in starts:
            win = g.iloc[i0:i0 + years]
            if win[col].isna().any():
                continue

            base = float(win[col].iloc[0])
            git = (win[col].to_numpy() / base) - 1.0
            t = np.arange(1, years + 1, dtype=float)
            dfactors = (1.0 + discount_rate) ** (-t)

            piit = float(np.sum(git * dfactors))
            net_pv_per_debt_stock = piit / af
            loss = min(net_pv_per_debt_stock, 0.0)

            out.append({
                "iso3": iso3,
                "country": win["country"].iloc[0],
                "window_start": int(win["year"].iloc[0]),
                "window_end": int(win["year"].iloc[-1]),
                "baseline": base,
                "net_pv_by_debt_stock": net_pv_per_debt_stock,
                "loss": loss,
            })

    return pd.DataFrame(out)


from scipy.optimize import brentq


def equivalent_rate(expected_loss, discount_rate, years=15):
    """Translate an expected PV loss (fraction of debt stock) into the
    equivalent interest rate.

    The expected loss is treated as additional debt, so the inflated principal
    is (1 + |expected_loss|). Solve for the rate r_new at which the annuity on
    the original principal (=1) equals the annuity on the inflated principal
    priced at the original rate, i.e.
        af(r_new, years) == af0 / (1 + |loss|),
    where `af0` is the original annuity factor. af is strictly decreasing in r,
    so the root is unique.
    """
    af0 = (1 - (1 + discount_rate) ** -years) / discount_rate
    target_af = af0 / (1.0 + abs(expected_loss))
    af = lambda r: (1.0 - (1.0 + r) ** -years) / r
    return brentq(lambda r: af(r) - target_af, 1e-9, 1.0)

In [ ]:
# --- Single-country view of the terminal residuals window by window ---
# Reuses compute_terminal_residuals_wdi for the residual numbers (and the per-window
# `baseline`, i.e. H at the window start). Top panel: H, with the selected 15-yr window
# shaded green where H sits above its start level (gain) and red where below (loss).
# Bottom panel: one bar per annual rolling window (= net_pv_by_debt_stock), plotted at
# the year the window matures (window_end + 1). Click a bar to shade its window above.

def country_residuals_chart(iso3, years=window_length, discount_rate=discount_rate,
                            step=1, first_start=start_year, grid=300):
    res = compute_terminal_residuals_wdi(
        df[df['iso3'] == iso3], years=years, discount_rate=discount_rate,
        window_mode="rolling", col="H", start_year=first_start)
    if step > 1:  # thin the rolling windows out if requested
        res = res[(res['window_start'] - res['window_start'].min()) % step == 0]
    res = res.reset_index(drop=True)

    g = (df[(df['iso3'] == iso3) & (df['year'] >= first_start)]
         .dropna(subset=['H']).sort_values('year'))
    country_name = g['country'].iloc[0]

    bars = res.assign(
        matures=res['window_end'] + 1,
        residual=res['net_pv_by_debt_stock'],
        sign=np.where(res['net_pv_by_debt_stock'] >= 0, 'gain', 'loss'),
        label=res['window_start'].astype(str) + '-' + res['window_end'].astype(str))

    # densify H within each window so the green/red split is clean where H crosses its start level
    shade_rows = []
    for _, r in res.iterrows():
        win = g[(g['year'] >= r['window_start']) & (g['year'] <= r['window_end'])]
        xs = np.linspace(win['year'].min(), win['year'].max(), grid)
        hs = np.interp(xs, win['year'], win['H'])
        for x, h in zip(xs, hs):
            shade_rows.append({'window_start': int(r['window_start']), 'year': x,
                               'ref': r['baseline'], 'above': max(h, r['baseline']),
                               'below': min(h, r['baseline'])})
    shade = pd.DataFrame(shade_rows)

    xscale = alt.Scale(domain=[int(g['year'].min()), int(g['year'].max()) + 1])
    sel = alt.selection_point(fields=['window_start'], on='click', empty=False,
                              value=[{'window_start': int(bars['window_start'].iloc[0])}])

    line = alt.Chart(g).mark_line(color='#222', strokeWidth=2).encode(
        x=alt.X('year:Q', scale=xscale, axis=alt.Axis(format='d', title=None)),
        y=alt.Y('H:Q', title='H = CPI / exchange_rate'))
    green = alt.Chart(shade).transform_filter(sel).mark_area(
        opacity=0.4, color='#2ca02c').encode(x='year:Q', y='ref:Q', y2='above:Q')
    red = alt.Chart(shade).transform_filter(sel).mark_area(
        opacity=0.4, color='#d62728').encode(x='year:Q', y='below:Q', y2='ref:Q')
    refrule = alt.Chart(shade).transform_filter(sel).mark_line(
        strokeDash=[4, 4], color='#444').encode(x='year:Q', y='ref:Q')
    top = (green + red + refrule + line).properties(
        width=660, height=300,
        title=alt.TitleParams(
            f"{country_name}: H with selectable {years}-yr window",
            subtitle="green = H above window start (gain) | red = below (loss); "
                     "click a bar to change window"))

    zero = alt.Chart(pd.DataFrame({'y': [0]})).mark_rule(color='#888').encode(y='y:Q')
    bar = alt.Chart(bars).mark_bar(width=10).encode(
        x=alt.X('matures:Q', scale=xscale,
                axis=alt.Axis(format='d', title=f"year window matures (start + {years})")),
        y=alt.Y('residual:Q', title='terminal residual / debt stock'),
        color=alt.Color('sign:N', scale=alt.Scale(domain=['gain', 'loss'],
                        range=['#2ca02c', '#d62728']), legend=None),
        opacity=alt.condition(sel, alt.value(1.0), alt.value(0.35)),
        tooltip=[alt.Tooltip('label:N', title='window'),
                 alt.Tooltip('residual:Q', format='+.4f')]).add_params(sel)
    bottom = (zero + bar).properties(width=660, height=170)

    return alt.vconcat(top, bottom).resolve_scale(x='shared').configure_view(strokeWidth=0)


country_residuals_chart('MAR')


In [ ]:
# --- Alternative: colour each window instead of shading one ---
# The top panel shows the hybrid index H re-based to 100 in the first period of the series
# (H/H0 * 100). Each rolling window is drawn as a horizontal segment at its own starting
# level (the index value at the window start), spanning its 15 years, shaded by a single
# hue (light = early, dark = late) for a sense of sequence; the bottom panel plots each
# window's residual (= net_pv_by_debt_stock) at the year the window ends (window_end) in
# the matching shade. The y-axis does not start at zero -- it is padded just enough to keep
# the whole series in view.

def country_windows_colored(iso3, years=window_length, discount_rate=discount_rate,
                            step=1, first_start=start_year, ypad_frac=0.05):
    res = compute_terminal_residuals_wdi(
        df[df['iso3'] == iso3], years=years, discount_rate=discount_rate,
        window_mode="rolling", col="H", start_year=first_start).reset_index(drop=True)
    if step > 1:  # thin the rolling windows out if requested
        res = res[(res['window_start'] - res['window_start'].min()) % step == 0].reset_index(drop=True)

    g = (df[(df['iso3'] == iso3) & (df['year'] >= first_start)]
         .dropna(subset=['H']).sort_values('year'))
    country_name = g['country'].iloc[0]
    h0 = float(g['H'].iloc[0])                       # first period of the whole series
    base_year = int(g['year'].iloc[0])

    g = g.assign(idx=g['H'] / h0 * 100.0)            # index = 100 in the first year
    wins = res.assign(residual=res['net_pv_by_debt_stock'],
                      start_idx=res['baseline'] / h0 * 100.0)  # window start level, as index

    xscale = alt.Scale(domain=[base_year, int(g['year'].max()) + 1])
    # y-axis: don't start at zero; pad just enough to keep the full series in view
    ypad = ypad_frac * (g['idx'].max() - g['idx'].min())
    yscale = alt.Scale(domain=[g['idx'].min() - ypad, g['idx'].max() + ypad],
                       zero=False, nice=False)
    # single hue, light (early) -> dark (late) so the window order reads as a gradient
    color = alt.Color('window_start:Q', scale=alt.Scale(range=['#9ecae1', '#08306b']),
                      legend=alt.Legend(title='window start', format='d'))
    tip = ['window_start', 'window_end', alt.Tooltip('residual:Q', format='+.4f')]

    ref100 = alt.Chart(pd.DataFrame({'y': [100]})).mark_rule(
        color='#bbb', strokeDash=[4, 4]).encode(y=alt.Y('y:Q', scale=yscale))
    path = alt.Chart(g).mark_line(color='#333', strokeWidth=2).encode(
        x=alt.X('year:Q', scale=xscale, axis=alt.Axis(format='d', title=None)),
        y=alt.Y('idx:Q', title=f'H index (=100 in {base_year})', scale=yscale),
        tooltip=['year', alt.Tooltip('idx:Q', format='.1f')])
    segs = alt.Chart(wins).mark_rule(strokeWidth=3, opacity=0.85).encode(
        x=alt.X('window_start:Q', scale=xscale), x2='window_end:Q',
        y=alt.Y('start_idx:Q', scale=yscale), color=color, tooltip=tip)
    top = (ref100 + segs + path).properties(
        width=660, height=300,
        title=alt.TitleParams(
            f"{country_name}: hybrid index H (=100 in {base_year})",
            subtitle="horizontal line = each 15-yr window's starting level; darker = later window"))

    zero = alt.Chart(pd.DataFrame({'y': [0]})).mark_rule(color='#888').encode(y='y:Q')
    bar = alt.Chart(wins).mark_bar(width=10).encode(
        x=alt.X('window_end:Q', scale=xscale,
                axis=alt.Axis(format='d', title=f"year window ends (start + {years - 1})")),
        y=alt.Y('residual:Q', title='terminal residual / debt stock'),
        color=color, tooltip=tip)
    bottom = (zero + bar).properties(width=660, height=170)
    return alt.vconcat(top, bottom).resolve_scale(x='shared', color='shared')


country_windows_colored('MAR')


In [ ]:
# --- Redesigned colored-windows chart (new version, for comparison with the cell above) ---
# Color = green (gain) / red (loss) with a light->dark shade by window start within each hue,
# used identically on each window's horizontal start-level marker, its end dot on the H line,
# and its residual bar below. No gridlines; a yearly x-axis (rotated -60) labelled on the
# bottom panel only; the dashed-100 (top) and zero (bottom) reference lines are the anchors.

def _ramp(t, light, dark):
    """Hex color interpolated between `light` (t=0) and `dark` (t=1) in RGB."""
    lo = tuple(int(light[i:i + 2], 16) for i in (1, 3, 5))
    hi = tuple(int(dark[i:i + 2], 16) for i in (1, 3, 5))
    return '#%02x%02x%02x' % tuple(round(a + (b - a) * t) for a, b in zip(lo, hi))


def country_windows_styled(iso3, years=window_length, discount_rate=discount_rate,
                           step=1, first_start=start_year, ypad_frac=0.05):
    res = compute_terminal_residuals_wdi(
        df[df['iso3'] == iso3], years=years, discount_rate=discount_rate,
        window_mode="rolling", col="H", start_year=first_start).reset_index(drop=True)
    if step > 1:  # thin the rolling windows out if requested
        res = res[(res['window_start'] - res['window_start'].min()) % step == 0].reset_index(drop=True)

    g = (df[(df['iso3'] == iso3) & (df['year'] >= first_start)]
         .dropna(subset=['H']).sort_values('year'))
    country_name = g['country'].iloc[0]
    h0 = float(g['H'].iloc[0])
    base_year, last_year = int(g['year'].iloc[0]), int(g['year'].max())

    g = g.assign(idx=g['H'] / h0 * 100.0)            # index = 100 in the first year
    idx_by_year = dict(zip(g['year'], g['idx']))

    # bivariate color: hue = sign of residual (green gain / red loss); within each hue,
    # lightness ranks the window by start date (light = earliest of that colour -> dark =
    # latest). Normalising per colour (not over all windows) keeps the gradient visible even
    # when one colour has only a few, tightly-clustered windows (e.g. a handful of late losses).
    s = res['window_start']
    sign_pos = res['net_pv_by_debt_stock'] >= 0

    def _shade(mask):
        sub = s[mask]
        if len(sub) == 0:
            return {}
        lo, sp = sub.min(), max(int(sub.max() - sub.min()), 1)
        return {int(ws): 0.25 + 0.6 * (ws - lo) / sp for ws in sub}

    gt, lt = _shade(sign_pos), _shade(~sign_pos)
    fills = [
        _ramp(gt[int(ws)], '#bae4b3', '#005a32') if resid >= 0
        else _ramp(lt[int(ws)], '#fcae91', '#a50f15')
        for ws, resid in zip(s, res['net_pv_by_debt_stock'])
    ]

    wins = res.assign(
        residual=res['net_pv_by_debt_stock'],
        start_idx=res['baseline'] / h0 * 100.0,
        end_idx=res['window_end'].map(idx_by_year),
        fill=fills)

    xscale = alt.Scale(domain=[base_year, last_year + 1])
    ypad = ypad_frac * (g['idx'].max() - g['idx'].min())
    yscale = alt.Scale(domain=[g['idx'].min() - ypad, g['idx'].max() + ypad],
                       zero=False, nice=False)
    year_vals = list(range(base_year, last_year + 1))
    fill_enc = alt.Color('fill:N', scale=None)
    tip = ['window_start', 'window_end', alt.Tooltip('residual:Q', format='+.4f')]

    # Layers share a scale per channel, so every layer must carry the SAME axis spec:
    # axis=None on any layer drops the guide, and differing fields with default titles make
    # Vega blank the title -- so define each axis once and reuse it everywhere.
    top_x = alt.Axis(values=year_vals, labels=False, ticks=True, title=None)  # ticks only
    top_y = alt.Axis(title=f'H index (=100 in {base_year})')
    bot_x = alt.Axis(values=year_vals, labelAngle=-60, format='d', title='year window ends')
    bot_y = alt.Axis(title='terminal residual / debt stock')

    # --- top panel: H index, window start-level markers + end dots (full-opacity so their
    #     gradient matches the residual bars below) ---
    ref100 = alt.Chart(pd.DataFrame({'y': [100]})).mark_rule(
        color='#bbb', strokeDash=[4, 4]).encode(y=alt.Y('y:Q', scale=yscale, axis=top_y))
    segs = alt.Chart(wins).mark_rule(strokeWidth=3).encode(
        x=alt.X('window_start:Q', scale=xscale, axis=top_x), x2='window_end:Q',
        y=alt.Y('start_idx:Q', scale=yscale, axis=top_y), color=fill_enc, tooltip=tip)
    hline = alt.Chart(g).mark_line(color='#222', strokeWidth=2).encode(
        x=alt.X('year:Q', scale=xscale, axis=top_x),
        y=alt.Y('idx:Q', scale=yscale, axis=top_y),
        tooltip=['year', alt.Tooltip('idx:Q', format='.1f')])
    dots = alt.Chart(wins).mark_point(filled=True, size=45).encode(
        x=alt.X('window_end:Q', scale=xscale, axis=top_x),
        y=alt.Y('end_idx:Q', scale=yscale, axis=top_y), color=fill_enc, tooltip=tip)
    top = (ref100 + segs + hline + dots).properties(
        width=720, height=300,
        title=alt.TitleParams(
            f"{country_name}: hybrid index H (=100 in {base_year})",
            subtitle="horizontal bar = each 15-yr window's start level, dot = where H ends; "
                     "green = gain, red = loss (darker = later window)",
            anchor='start'))

    # --- bottom panel: terminal residual per window, at window end ---
    zero = alt.Chart(pd.DataFrame({'y': [0]})).mark_rule(color='#888').encode(
        y=alt.Y('y:Q', axis=bot_y))
    bars = alt.Chart(wins).mark_bar(width=10).encode(
        x=alt.X('window_end:Q', scale=xscale, axis=bot_x),
        y=alt.Y('residual:Q', axis=bot_y), color=fill_enc, tooltip=tip)
    bottom = (zero + bars).properties(width=720, height=160)

    return (alt.vconcat(top, bottom).resolve_scale(x='shared')
            .configure_view(strokeWidth=0)
            .configure_axis(grid=False, domainColor='#888', tickColor='#888'))


country_windows_styled('LSO')


In [ ]:
residuals = compute_terminal_residuals_wdi(df, years=window_length, discount_rate=discount_rate, window_mode="rolling", col="H", start_year=start_year)
af = (1 - (1 + discount_rate) ** -window_length) / discount_rate
expected_loss = residuals["loss"].mean()
prob_loss = (residuals["net_pv_by_debt_stock"] < 0).mean()
n_windows = len(residuals)
n_countries = residuals["iso3"].nunique()
r_new = equivalent_rate(expected_loss=expected_loss, discount_rate=discount_rate, years=window_length)


print(f"Windows from {start_year} to 2024")
print(f"Number of windows  : {n_windows}")
print(f"Number of countries: {n_countries}")
print(f"Expected loss      : {expected_loss:.4f}")
print(f"Probability of loss: {prob_loss:.4f}")
print(f"interest rate increase: {r_new-discount_rate}")
print(f"annual premium: {expected_loss/af} times debt stock")

hist = alt.Chart(residuals).mark_bar().encode(
    x=alt.X('net_pv_by_debt_stock:Q', bin=alt.Bin(maxbins=40), title='Terminal residual / Total Debt'),
    y=alt.Y('count():Q', title=f'Number of {window_length} year windows in pool'),
    color=alt.condition(
        alt.datum.net_pv_by_debt_stock < 0,
        alt.value('crimson'),
        alt.value('steelblue'),
    ),
).properties(
    width=600,
    height=350,
    title=f'Distribution of terminal residuals (n={n_windows}, expected loss={expected_loss:.3f}, P(loss)={prob_loss:.2%})'
)
hist

In [ ]:
by_country = (
    residuals.groupby(["iso3", "country"], as_index=False)
    .agg(
        expected_loss=("loss", "mean"),
        prob_loss=("net_pv_by_debt_stock", lambda s: (s < 0).mean()),
        n_windows=("loss", "size"),
    )
    .sort_values("expected_loss")  # most negative first = highest expected loss
    .reset_index(drop=True)
)
# Equivalent rate increase: the additional interest rate (over the original
# discount_rate) that would impose the same PV burden as the expected loss.
by_country["rate_increase"] = by_country["expected_loss"].apply(
    lambda el: equivalent_rate(el, discount_rate=discount_rate, years=window_length) - discount_rate
)
# expected_loss is net PV normalized by the annuity factor (af), i.e. a fraction
# of the annuity. Dividing by af again expresses it as a fraction of debt stock.
by_country["expected_loss_debt"] = by_country["expected_loss"] / af
print(by_country.to_string(index=False))

sort_order = by_country["country"].tolist()
shared_tooltip = ["country", alt.Tooltip("expected_loss:Q", format=".4f"),
                  alt.Tooltip("expected_loss_debt:Q", format=".4f"),
                  alt.Tooltip("prob_loss:Q", format=".2%"),
                  alt.Tooltip("rate_increase:Q", format=".4f"),
                  "n_windows"]

# Left axis: fraction of the annuity (the bars actually drawn).
bar_annuity = alt.Chart(by_country).mark_bar(color="crimson").encode(
    x=alt.X("country:N", sort=sort_order, title="Country", axis=alt.Axis(labelAngle=-60)),
    y=alt.Y("expected_loss:Q", title="Expected loss (fraction of annuity)"),
    tooltip=shared_tooltip,
)

# Right axis: same loss expressed as a fraction of the debt stock (= /af).
# Drawn invisibly so it only contributes the secondary axis.
bar_debt = alt.Chart(by_country).mark_bar(opacity=0).encode(
    x=alt.X("country:N", sort=sort_order),
    y=alt.Y("expected_loss_debt:Q",
            title="Expected loss (fraction of debt stock)",
            axis=alt.Axis(orient="right")),
    tooltip=shared_tooltip,
)

bar = alt.layer(bar_annuity, bar_debt).resolve_scale(y="independent").properties(
    width=800,
    height=400,
    title="Expected loss by country (sorted: highest expected loss → lowest)",
)
bar

In [ ]:
rate_sorted = by_country.sort_values("rate_increase", ascending=False).reset_index(drop=True)

rate_bar = alt.Chart(rate_sorted).mark_bar().encode(
    x=alt.X("country:N", sort=rate_sorted["country"].tolist(), title="Country", axis=alt.Axis(labelAngle=-60)),
    y=alt.Y("rate_increase:Q", title="Equivalent rate increase (over original interest rate)",
            axis=alt.Axis(format="%")),
    tooltip=["country", alt.Tooltip("rate_increase:Q", format=".4f"),
             alt.Tooltip("expected_loss:Q", format=".4f"),
             alt.Tooltip("prob_loss:Q", format=".2%"),
             "n_windows"],
    color=alt.value("darkorange"),
).properties(
    width=800,
    height=400,
    title="Equivalent rate increase by country (sorted: highest → lowest)",
)
rate_bar

In [ ]:
af

In [ ]:
import math

countries = df[['iso3', 'country']].drop_duplicates().sort_values('country').reset_index(drop=True)

charts = []
for i, row in countries.iterrows():
    iso3, country_name = row['iso3'], row['country']
    cdata = df[(df['iso3'] == iso3) & (df['year'] >= start_year)].dropna(subset=['usd_per_lcu', 'H'])
    if cdata.empty:
        continue

    legend_setting = alt.Legend(title="Series", orient="bottom") if i % 2 == 0 else None

    left_chart = alt.Chart(cdata).mark_line(point=True).encode(
        x=alt.X('year:O', title='Year'),
        y=alt.Y('usd_per_lcu:Q', title='USD per LCU'),
        color=alt.Color('Metric:N', legend=legend_setting),
        tooltip=['year', 'usd_per_lcu']
    ).transform_calculate(Metric="'usd_per_lcu'")

    right_chart = alt.Chart(cdata).mark_line(point=True, color='orange').encode(
        x=alt.X('year:O', title='Year'),
        y=alt.Y('H:Q', title='H = CPI / exchange_rate', axis=alt.Axis(orient='right')),
        color=alt.Color('Metric:N', legend=legend_setting),
        tooltip=['year', 'H']
    ).transform_calculate(Metric="'H'")

    base = alt.layer(left_chart, right_chart).resolve_scale(y='independent').properties(
        title=country_name,
        width=400,
        height=300
    )
    charts.append(base)

def chart_list_to_rows_of_two(chart_list):
    rows = math.ceil(len(chart_list) / 2)
    chart_rows = []
    for i in range(rows):
        charts_in_row = chart_list[i*2 : i*2 + 2]
        if len(charts_in_row) == 2:
            r = alt.hconcat(*charts_in_row).resolve_scale(color='independent')
        else:
            r = charts_in_row[0]
        chart_rows.append(r)
    return alt.vconcat(*chart_rows)

final_layout = chart_list_to_rows_of_two(charts)
final_layout